## Imports and project paths

In [1]:
# ============================================================
# 1. Imports and project paths
# ============================================================

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


BASE_DIR = Path.cwd().parent

DATA_PATH = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "earlysignal_monitoring_data.csv"
)

RISK_MODEL_PATH = (
    BASE_DIR
    / "models"
    / "earlysignal_risk_model.joblib"
)

RISK_RESULTS_PATH = (
    BASE_DIR
    / "outputs"
    / "risk_results.csv"
)

OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 65)
print("EARLYSIGNAL AI — EXPLAINABLE AI SETUP")
print("=" * 65)

print("\nBase directory:")
print(BASE_DIR)

print("\nDataset exists:")
print(DATA_PATH.exists())

print("\nRisk model exists:")
print(RISK_MODEL_PATH.exists())

print("\nRisk results exist:")
print(RISK_RESULTS_PATH.exists())

EARLYSIGNAL AI — EXPLAINABLE AI SETUP

Base directory:
C:\Users\Ezekiel Mbaya\myenv\EarlySignal_AI

Dataset exists:
True

Risk model exists:
True

Risk results exist:
True


## Load data and trained Risk Intelligence model

In [2]:
# ============================================================
# 2. Load monitoring data, risk results and trained model
# ============================================================

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["reporting_date"]
)

risk_results = pd.read_csv(
    RISK_RESULTS_PATH,
    parse_dates=["reporting_date"]
)

risk_model = joblib.load(
    RISK_MODEL_PATH
)


print("=" * 65)
print("EXPLAINABILITY INPUTS")
print("=" * 65)

print(
    "\nMonitoring dataset:",
    df.shape
)

print(
    "Risk results:",
    risk_results.shape
)

print(
    "\nRisk model type:",
    type(risk_model).__name__
)

print(
    "Risk pipeline steps:",
    list(risk_model.named_steps.keys())
)

EXPLAINABILITY INPUTS

Monitoring dataset: (7680, 22)
Risk results: (1680, 14)

Risk model type: Pipeline
Risk pipeline steps: ['preprocessor', 'model']


## Re-establish the Risk Intelligence feature contract

In [3]:
# ============================================================
# 3. Risk Intelligence feature contract
# ============================================================

numeric_features = [
    "achievement_rate",
    "activity_completion_rate",
    "budget_utilisation_rate",
    "reporting_delay_days",
    "complaints_count",
    "staff_availability_rate",
    "supply_delay_days",
    "previous_month_achievement_rate",
    "access_constraint_score",
    "data_quality_score"
]

categorical_features = [
    "programme_name",
    "sector",
    "state",
    "lga",
    "community"
]

risk_features = (
    numeric_features
    + categorical_features
)

forbidden_explanation_inputs = [
    "risk_label",
    "recommended_action",
    "anomaly_flag",
    "monthly_achievement",
    "record_id"
]

leakage_overlap = set(
    risk_features
).intersection(
    forbidden_explanation_inputs
)


print("=" * 65)
print("EXPLAINABILITY FEATURE CONTRACT")
print("=" * 65)

print(
    "\nNumeric features:",
    len(numeric_features)
)

print(
    "Categorical features:",
    len(categorical_features)
)

print(
    "Total risk features:",
    len(risk_features)
)

print(
    "\nForbidden inputs found:",
    leakage_overlap
)

print(
    "\nExplainability leakage audit:",
    "PASSED"
    if len(leakage_overlap) == 0
    else "FAILED"
)

EXPLAINABILITY FEATURE CONTRACT

Numeric features: 10
Categorical features: 5
Total risk features: 15

Forbidden inputs found: set()

Explainability leakage audit: PASSED


## Inspect the fitted pipeline

In [4]:
# ============================================================
# 4. Inspect fitted Risk Intelligence pipeline
# ============================================================

risk_preprocessor = (
    risk_model.named_steps[
        "preprocessor"
    ]
)

risk_classifier = (
    risk_model.named_steps[
        "model"
    ]
)

transformed_feature_names = (
    risk_preprocessor
    .get_feature_names_out()
)

model_classes = (
    risk_classifier.classes_
)


print("=" * 65)
print("TRAINED RISK MODEL STRUCTURE")
print("=" * 65)

print(
    "\nTransformed features:",
    len(transformed_feature_names)
)

print(
    "Model classes:",
    model_classes
)

print(
    "\nCoefficient matrix shape:",
    risk_classifier.coef_.shape
)

print(
    "Intercept shape:",
    risk_classifier.intercept_.shape
)

print(
    "\nFirst 15 transformed features:"
)

for feature in transformed_feature_names[:15]:
    print(
        " -",
        feature
    )

TRAINED RISK MODEL STRUCTURE

Transformed features: 77
Model classes: ['High' 'Low' 'Medium']

Coefficient matrix shape: (3, 77)
Intercept shape: (3,)

First 15 transformed features:
 - numeric__achievement_rate
 - numeric__activity_completion_rate
 - numeric__budget_utilisation_rate
 - numeric__reporting_delay_days
 - numeric__complaints_count
 - numeric__staff_availability_rate
 - numeric__supply_delay_days
 - numeric__previous_month_achievement_rate
 - numeric__access_constraint_score
 - numeric__data_quality_score
 - categorical__programme_name_Community Resilience Programme
 - categorical__programme_name_Education Access Project
 - categorical__programme_name_Livelihood Recovery Initiative
 - categorical__programme_name_Nutrition Support Programme
 - categorical__programme_name_WASH Resilience Initiative


## Build global class coefficient table

In [5]:
# ============================================================
# 5. Build global coefficient table
# ============================================================

coefficient_table = pd.DataFrame(
    risk_classifier.coef_.T,
    index=transformed_feature_names,
    columns=model_classes
)

coefficient_table.index.name = (
    "transformed_feature"
)


print("=" * 70)
print("GLOBAL RISK COEFFICIENT TABLE")
print("=" * 70)

print(
    "\nShape:",
    coefficient_table.shape
)

display(
    coefficient_table.head(15)
)

GLOBAL RISK COEFFICIENT TABLE

Shape: (77, 3)


,High,Low,Medium
transformed_feature,,,
numeric__achievement_rate,-8.147930,9.091605,-0.943674
numeric__activity_completion_rate,-3.653117,4.168645,-0.515528
numeric__budget_utilisation_rate,0.034583,-0.005769,-0.028814
numeric__reporting_delay_days,4.530539,-5.314733,0.784194
numeric__complaints_count,2.259691,-2.582083,0.322392
numeric__staff_availability_rate,-0.785780,0.910007,-0.124227
numeric__supply_delay_days,3.937114,-4.455426,0.518311
numeric__previous_month_achievement_rate,0.133554,-0.019270,-0.114284
numeric__access_constraint_score,1.545217,-1.842624,0.297407


## Global High-Risk drivers

In [6]:
# ============================================================
# 6. Global High-Risk model drivers
# ============================================================

high_class_coefficients = (
    coefficient_table["High"]
    .sort_values(
        ascending=False
    )
)

top_high_drivers = (
    high_class_coefficients
    .head(15)
    .rename(
        "high_risk_coefficient"
    )
    .to_frame()
)

strongest_away_from_high = (
    high_class_coefficients
    .tail(15)
    .sort_values()
    .rename(
        "high_risk_coefficient"
    )
    .to_frame()
)


print("=" * 70)
print("GLOBAL HIGH-RISK DRIVERS")
print("=" * 70)

print(
    "\nStrongest signals toward High Risk:"
)

display(
    top_high_drivers
)

print(
    "\nStrongest signals away from High Risk:"
)

display(
    strongest_away_from_high
)

print(
    "\nInterpretation note:"
)

print(
    "Numeric variables were standardized and categorical "
    "variables one-hot encoded. Coefficient magnitude should "
    "therefore be interpreted within the transformed model "
    "rather than as original measurement units."
)

GLOBAL HIGH-RISK DRIVERS

Strongest signals toward High Risk:


,high_risk_coefficient
transformed_feature,
numeric__reporting_delay_days,4.530539
numeric__supply_delay_days,3.937114
numeric__complaints_count,2.259691
numeric__access_constraint_score,1.545217
categorical__community_Gwoza Site 2,0.813749
categorical__community_Dikwa Site 2,0.690736
categorical__community_Maiduguri Site 1,0.660034
categorical__community_Mafa Site 6,0.587808
categorical__community_Bama Site 4,0.537355



Strongest signals away from High Risk:


,high_risk_coefficient
transformed_feature,
numeric__achievement_rate,-8.147930
numeric__activity_completion_rate,-3.653117
categorical__community_Dikwa Site 4,-0.924102
numeric__staff_availability_rate,-0.785780
categorical__community_Maiduguri Site 5,-0.646588
categorical__community_Jere Site 3,-0.632938
categorical__community_Konduga Site 2,-0.546927
categorical__community_Maiduguri Site 6,-0.432781
categorical__community_Maiduguri Site 2,-0.409511



Interpretation note:
Numeric variables were standardized and categorical variables one-hot encoded. Coefficient magnitude should therefore be interpreted within the transformed model rather than as original measurement units.


## Reconstruct the held-out explanation dataset

In [7]:
# ============================================================
# 7. Reconstruct held-out explanation dataset
# ============================================================

explanation_df = (
    risk_results[
        [
            "record_id",
            "reporting_date",
            "programme_name",
            "sector",
            "state",
            "lga",
            "community",
            "risk_label",
            "predicted_risk",
            "probability_high",
            "probability_low",
            "probability_medium"
        ]
    ]
    .merge(
        df[
            [
                "record_id",
                "achievement_rate",
                "activity_completion_rate",
                "budget_utilisation_rate",
                "reporting_delay_days",
                "complaints_count",
                "staff_availability_rate",
                "supply_delay_days",
                "previous_month_achievement_rate",
                "access_constraint_score",
                "data_quality_score"
            ]
        ],
        on="record_id",
        how="left",
        validate="one_to_one"
    )
)

print("=" * 70)
print("HELD-OUT EXPLANATION DATASET")
print("=" * 70)

print(
    "\nRows:",
    len(explanation_df)
)

print(
    "Columns:",
    len(explanation_df.columns)
)

print(
    "Missing values:",
    int(explanation_df.isna().sum().sum())
)

print(
    "Duplicate record IDs:",
    int(explanation_df["record_id"].duplicated().sum())
)

print("\nPredicted risk distribution:")

print(
    explanation_df[
        "predicted_risk"
    ].value_counts()
)

HELD-OUT EXPLANATION DATASET

Rows: 1680
Columns: 22
Missing values: 0
Duplicate record IDs: 0

Predicted risk distribution:
predicted_risk
Low       925
Medium    615
High      140
Name: count, dtype: int64


## Verify predictions against the loaded model

In [8]:
# ============================================================
# 8. Verify explanation data against trained model
# ============================================================

X_explain = explanation_df[
    risk_features
].copy()

recomputed_predictions = (
    risk_model.predict(
        X_explain
    )
)

recomputed_probabilities = (
    risk_model.predict_proba(
        X_explain
    )
)

prediction_match = (
    recomputed_predictions
    == explanation_df[
        "predicted_risk"
    ].to_numpy()
)

print("=" * 70)
print("EXPLAINABILITY REPRODUCIBILITY CHECK")
print("=" * 70)

print(
    "\nPredictions checked:",
    len(prediction_match)
)

print(
    "Matching predictions:",
    int(prediction_match.sum())
)

print(
    "Prediction match rate:",
    f"{prediction_match.mean():.2%}"
)

print(
    "\nModel classes:",
    risk_classifier.classes_
)

EXPLAINABILITY REPRODUCIBILITY CHECK

Predictions checked: 1680
Matching predictions: 1680
Prediction match rate: 100.00%

Model classes: ['High' 'Low' 'Medium']


## Calculate record-level feature contributions

In [9]:
# ============================================================
# 9. Calculate record-level model contributions
# ============================================================

X_transformed = (
    risk_preprocessor.transform(
        X_explain
    )
)

if hasattr(
    X_transformed,
    "toarray"
):
    X_transformed = (
        X_transformed.toarray()
    )

high_class_index = (
    list(model_classes).index(
        "High"
    )
)

high_coefficients = (
    risk_classifier.coef_[
        high_class_index
    ]
)

high_contributions = (
    X_transformed
    * high_coefficients
)

high_contribution_df = pd.DataFrame(
    high_contributions,
    columns=transformed_feature_names,
    index=explanation_df.index
)

print("=" * 70)
print("RECORD-LEVEL HIGH-RISK CONTRIBUTIONS")
print("=" * 70)

print(
    "\nContribution matrix:",
    high_contribution_df.shape
)

print(
    "Expected rows:",
    len(explanation_df)
)

print(
    "Transformed features:",
    len(transformed_feature_names)
)

print(
    "\nHigh-class intercept:",
    round(
        float(
            risk_classifier.intercept_[
                high_class_index
            ]
        ),
        4
    )
)

RECORD-LEVEL HIGH-RISK CONTRIBUTIONS

Contribution matrix: (1680, 77)
Expected rows: 1680
Transformed features: 77

High-class intercept: -9.4588


## Mathematical contribution audit

In [10]:
# ============================================================
# 10. Mathematical contribution audit
# ============================================================

reconstructed_high_score = (
    high_contribution_df.sum(
        axis=1
    ).to_numpy()
    + risk_classifier.intercept_[
        high_class_index
    ]
)

model_decision_scores = (
    risk_classifier.decision_function(
        X_transformed
    )
)

model_high_score = (
    model_decision_scores[
        :,
        high_class_index
    ]
)

maximum_score_difference = (
    np.max(
        np.abs(
            reconstructed_high_score
            - model_high_score
        )
    )
)

print("=" * 70)
print("EXPLANATION MATHEMATICAL AUDIT")
print("=" * 70)

print(
    "\nMaximum reconstruction difference:",
    f"{maximum_score_difference:.12f}"
)

print(
    "\nContribution reconstruction:",
    "PASSED"
    if maximum_score_difference < 1e-10
    else "CHECK REQUIRED"
)

EXPLANATION MATHEMATICAL AUDIT

Maximum reconstruction difference: 0.000000000000

Contribution reconstruction: PASSED


## Map transformed numeric features to manager-friendly names

In [11]:
# ============================================================
# 11. Manager-friendly operational feature names
# ============================================================

friendly_feature_names = {
    "numeric__achievement_rate":
        "Achievement rate",

    "numeric__activity_completion_rate":
        "Activity completion",

    "numeric__budget_utilisation_rate":
        "Budget utilisation",

    "numeric__reporting_delay_days":
        "Reporting delay",

    "numeric__complaints_count":
        "Complaints",

    "numeric__staff_availability_rate":
        "Staff availability",

    "numeric__supply_delay_days":
        "Supply delay",

    "numeric__previous_month_achievement_rate":
        "Previous-month achievement",

    "numeric__access_constraint_score":
        "Access constraints",

    "numeric__data_quality_score":
        "Data quality"
}

operational_transformed_features = list(
    friendly_feature_names.keys()
)

print("=" * 70)
print("MANAGER-FACING EXPLANATION FEATURES")
print("=" * 70)

print(
    "\nOperational explanation features:",
    len(operational_transformed_features)
)

for feature in operational_transformed_features:
    print(
        f" - {feature} -> "
        f"{friendly_feature_names[feature]}"
    )

MANAGER-FACING EXPLANATION FEATURES

Operational explanation features: 10
 - numeric__achievement_rate -> Achievement rate
 - numeric__activity_completion_rate -> Activity completion
 - numeric__budget_utilisation_rate -> Budget utilisation
 - numeric__reporting_delay_days -> Reporting delay
 - numeric__complaints_count -> Complaints
 - numeric__staff_availability_rate -> Staff availability
 - numeric__supply_delay_days -> Supply delay
 - numeric__previous_month_achievement_rate -> Previous-month achievement
 - numeric__access_constraint_score -> Access constraints
 - numeric__data_quality_score -> Data quality


## Inspect local explanation for highest High-Risk case

In [12]:
# ============================================================
# 12. Inspect a high-priority local explanation
# ============================================================

highest_risk_index = (
    explanation_df[
        "probability_high"
    ].idxmax()
)

highest_risk_record = (
    explanation_df.loc[
        highest_risk_index
    ]
)

record_contributions = (
    high_contribution_df.loc[
        highest_risk_index,
        operational_transformed_features
    ]
)

local_explanation = pd.DataFrame({
    "Operational Driver": [
        friendly_feature_names[
            feature
        ]
        for feature
        in operational_transformed_features
    ],

    "Contribution to High Risk": [
        record_contributions[
            feature
        ]
        for feature
        in operational_transformed_features
    ]
})

local_explanation[
    "Direction"
] = np.where(
    local_explanation[
        "Contribution to High Risk"
    ] > 0,
    "Toward High Risk",
    "Away from High Risk"
)

local_explanation = (
    local_explanation
    .sort_values(
        "Contribution to High Risk",
        ascending=False
    )
    .reset_index(drop=True)
)


print("=" * 75)
print("EXAMPLE — LOCAL HIGH-RISK EXPLANATION")
print("=" * 75)

print(
    "\nRecord ID:",
    highest_risk_record[
        "record_id"
    ]
)

print(
    "Programme:",
    highest_risk_record[
        "programme_name"
    ]
)

print(
    "Location:",
    highest_risk_record[
        "lga"
    ],
    "-",
    highest_risk_record[
        "community"
    ]
)

print(
    "Predicted Risk:",
    highest_risk_record[
        "predicted_risk"
    ]
)

print(
    "Model High-Risk Score:",
    f"{highest_risk_record['probability_high']:.4f}"
)

print(
    "\nOperational contributions:"
)

display(
    local_explanation.round(4)
)

print(
    "\nInterpretation note: positive contributions "
    "push the model toward the High-Risk class; "
    "negative contributions push it away."
)

EXAMPLE — LOCAL HIGH-RISK EXPLANATION

Record ID: ES-006044
Programme: Nutrition Support Programme
Location: Gwoza - Gwoza Site 3
Predicted Risk: High
Model High-Risk Score: 1.0000

Operational contributions:


,Operational Driver,Contribution to High Risk,Direction
0,Supply delay,46.6479,Toward High Risk
1,Reporting delay,10.7786,Toward High Risk
2,Achievement rate,3.1252,Toward High Risk
3,Complaints,0.7850,Toward High Risk
4,Staff availability,0.3123,Toward High Risk
5,Data quality,0.0721,Toward High Risk
6,Activity completion,0.0655,Toward High Risk
7,Previous-month achievement,0.0451,Toward High Risk
8,Budget utilisation,0.0187,Toward High Risk
9,Access constraints,-1.3129,Away from High Risk



Interpretation note: positive contributions push the model toward the High-Risk class; negative contributions push it away.


## Define operational explanation language

In [13]:
# ============================================================
# 13. Define manager-friendly explanation language
# ============================================================

driver_language = {
    "achievement_rate": {
        "toward": "Low achievement",
        "away": "Strong achievement"
    },

    "activity_completion_rate": {
        "toward": "Low activity completion",
        "away": "Strong activity completion"
    },

    "budget_utilisation_rate": {
        "toward": "Budget utilisation pattern",
        "away": "Budget utilisation pattern"
    },

    "reporting_delay_days": {
        "toward": "Reporting delay",
        "away": "Timely reporting"
    },

    "complaints_count": {
        "toward": "Elevated complaints",
        "away": "Lower complaints"
    },

    "staff_availability_rate": {
        "toward": "Low staff availability",
        "away": "Strong staff availability"
    },

    "supply_delay_days": {
        "toward": "Supply delay",
        "away": "Limited supply delay"
    },

    "previous_month_achievement_rate": {
        "toward": "Previous-month achievement pattern",
        "away": "Previous-month achievement pattern"
    },

    "access_constraint_score": {
        "toward": "Access constraints",
        "away": "Lower access constraints"
    },

    "data_quality_score": {
        "toward": "Lower data quality",
        "away": "Strong data quality"
    }
}


print("=" * 70)
print("MANAGER-FRIENDLY EXPLANATION LANGUAGE")
print("=" * 70)

for feature, labels in driver_language.items():
    print(
        f"\n{feature}"
    )
    print(
        " Toward High Risk:",
        labels["toward"]
    )
    print(
        " Away from High Risk:",
        labels["away"]
    )

MANAGER-FRIENDLY EXPLANATION LANGUAGE

achievement_rate
 Toward High Risk: Low achievement
 Away from High Risk: Strong achievement

activity_completion_rate
 Toward High Risk: Low activity completion
 Away from High Risk: Strong activity completion

budget_utilisation_rate
 Toward High Risk: Budget utilisation pattern
 Away from High Risk: Budget utilisation pattern

reporting_delay_days
 Toward High Risk: Reporting delay
 Away from High Risk: Timely reporting

complaints_count
 Toward High Risk: Elevated complaints
 Away from High Risk: Lower complaints

staff_availability_rate
 Toward High Risk: Low staff availability
 Away from High Risk: Strong staff availability

supply_delay_days
 Toward High Risk: Supply delay
 Away from High Risk: Limited supply delay

previous_month_achievement_rate
 Toward High Risk: Previous-month achievement pattern
 Away from High Risk: Previous-month achievement pattern

access_constraint_score
 Toward High Risk: Access constraints
 Away from High Risk: 

## Create local operational explanation function

In [14]:
# ============================================================
# 14. Local operational explanation function
# ============================================================

numeric_transformed_map = {
    feature:
        f"numeric__{feature}"
    for feature in numeric_features
}


def explain_high_risk_record(
    row_index,
    top_n=3
):
    contributions = []

    for original_feature in numeric_features:

        transformed_feature = (
            numeric_transformed_map[
                original_feature
            ]
        )

        contribution = float(
            high_contribution_df.loc[
                row_index,
                transformed_feature
            ]
        )

        direction = (
            "toward"
            if contribution > 0
            else "away"
        )

        label = (
            driver_language[
                original_feature
            ][direction]
        )

        contributions.append({
            "feature":
                original_feature,

            "driver":
                label,

            "contribution":
                contribution,

            "direction":
                direction
        })

    contribution_df = pd.DataFrame(
        contributions
    )

    toward_high = (
        contribution_df[
            contribution_df[
                "contribution"
            ] > 0
        ]
        .sort_values(
            "contribution",
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    away_from_high = (
        contribution_df[
            contribution_df[
                "contribution"
            ] < 0
        ]
        .assign(
            absolute_contribution=lambda x:
                x["contribution"].abs()
        )
        .sort_values(
            "absolute_contribution",
            ascending=False
        )
        .head(top_n)
        .drop(
            columns=[
                "absolute_contribution"
            ]
        )
        .reset_index(drop=True)
    )

    return (
        toward_high,
        away_from_high
    )


print(
    "Local explanation function created successfully."
)

Local explanation function created successfully.


## Test the explanation function

In [15]:
# ============================================================
# 15. Test manager-facing local explanation
# ============================================================

toward_high, away_from_high = (
    explain_high_risk_record(
        highest_risk_index,
        top_n=3
    )
)


print("=" * 75)
print("MANAGER-FACING LOCAL EXPLANATION")
print("=" * 75)

print(
    "\nRecord:",
    highest_risk_record[
        "record_id"
    ]
)

print(
    "Programme:",
    highest_risk_record[
        "programme_name"
    ]
)

print(
    "Location:",
    highest_risk_record[
        "lga"
    ],
    "-",
    highest_risk_record[
        "community"
    ]
)

print(
    "Predicted Risk:",
    highest_risk_record[
        "predicted_risk"
    ]
)

print(
    "\nMain signals toward High Risk:"
)

display(
    toward_high[
        [
            "driver",
            "contribution"
        ]
    ].round(4)
)

print(
    "\nMain signals reducing the High-Risk signal:"
)

display(
    away_from_high[
        [
            "driver",
            "contribution"
        ]
    ].round(4)
)

MANAGER-FACING LOCAL EXPLANATION

Record: ES-006044
Programme: Nutrition Support Programme
Location: Gwoza - Gwoza Site 3
Predicted Risk: High

Main signals toward High Risk:


,driver,contribution
0,Supply delay,46.6479
1,Reporting delay,10.7786
2,Low achievement,3.1252



Main signals reducing the High-Risk signal:


,driver,contribution
0,Lower access constraints,-1.3129


## Define investigation guidance

In [16]:
# ============================================================
# 16. Define investigation guidance
# ============================================================

investigation_guidance = {
    "achievement_rate":
        "Review progress against targets and identify implementation bottlenecks.",

    "activity_completion_rate":
        "Review delayed or incomplete activities and implementation constraints.",

    "budget_utilisation_rate":
        "Review expenditure patterns against implementation progress.",

    "reporting_delay_days":
        "Validate reporting timeliness and follow up on delayed field submissions.",

    "complaints_count":
        "Review feedback and complaint records for recurring operational issues.",

    "staff_availability_rate":
        "Review staffing coverage, absences and field deployment capacity.",

    "supply_delay_days":
        "Review procurement, logistics and supply-chain bottlenecks.",

    "previous_month_achievement_rate":
        "Review recent performance trends for persistent underachievement.",

    "access_constraint_score":
        "Review access, movement and operational constraints affecting delivery.",

    "data_quality_score":
        "Validate source data, completeness and reporting quality."
}


print("=" * 70)
print("INVESTIGATION GUIDANCE LIBRARY")
print("=" * 70)

for feature, guidance in investigation_guidance.items():
    print(
        f"\n{feature}:"
    )
    print(
        guidance
    )

INVESTIGATION GUIDANCE LIBRARY

achievement_rate:
Review progress against targets and identify implementation bottlenecks.

activity_completion_rate:
Review delayed or incomplete activities and implementation constraints.

budget_utilisation_rate:
Review expenditure patterns against implementation progress.

reporting_delay_days:
Validate reporting timeliness and follow up on delayed field submissions.

complaints_count:
Review feedback and complaint records for recurring operational issues.

staff_availability_rate:
Review staffing coverage, absences and field deployment capacity.

supply_delay_days:
Review procurement, logistics and supply-chain bottlenecks.

previous_month_achievement_rate:
Review recent performance trends for persistent underachievement.

access_constraint_score:
Review access, movement and operational constraints affecting delivery.

data_quality_score:
Validate source data, completeness and reporting quality.


## Generate a complete EarlySignal explanation

In [17]:
# ============================================================
# 17. Generate complete EarlySignal explanation
# ============================================================

def generate_earlysignal_explanation(
    row_index,
    top_n=3
):
    record = explanation_df.loc[
        row_index
    ]

    toward_high, away_from_high = (
        explain_high_risk_record(
            row_index,
            top_n=top_n
        )
    )

    main_drivers = (
        toward_high[
            "driver"
        ].tolist()
    )

    investigation_actions = []

    for feature in toward_high[
        "feature"
    ].tolist():

        guidance = (
            investigation_guidance[
                feature
            ]
        )

        if guidance not in investigation_actions:
            investigation_actions.append(
                guidance
            )

    if main_drivers:
        reason_text = (
            "The strongest operational signals "
            "associated with the High-Risk model "
            "score are "
            + ", ".join(main_drivers[:-1])
            + (
                " and "
                + main_drivers[-1]
                if len(main_drivers) > 1
                else main_drivers[-1]
            )
            + "."
        )
    else:
        reason_text = (
            "No strong operational feature contribution "
            "toward High Risk was identified."
        )

    return {
        "record_id":
            record["record_id"],

        "programme":
            record["programme_name"],

        "location":
            f"{record['lga']} - "
            f"{record['community']}",

        "predicted_risk":
            record["predicted_risk"],

        "high_risk_model_score":
            float(
                record[
                    "probability_high"
                ]
            ),

        "main_drivers":
            main_drivers,

        "reason":
            reason_text,

        "recommended_investigation":
            investigation_actions,

        "decision_use":
            "Recommended for human review; "
            "the model does not make operational decisions."
    }


example_explanation = (
    generate_earlysignal_explanation(
        highest_risk_index
    )
)


print("=" * 75)
print("EARLYSIGNAL AI — EXPLAINED RISK ALERT")
print("=" * 75)

print(
    "\nRecord ID:",
    example_explanation[
        "record_id"
    ]
)

print(
    "Programme:",
    example_explanation[
        "programme"
    ]
)

print(
    "Location:",
    example_explanation[
        "location"
    ]
)

print(
    "Predicted Risk:",
    example_explanation[
        "predicted_risk"
    ]
)

print(
    "High-Risk Model Score:",
    f"{example_explanation['high_risk_model_score']:.4f}"
)

print(
    "\nWhy this record was flagged:"
)

print(
    example_explanation[
        "reason"
    ]
)

print(
    "\nRecommended investigation:"
)

for action in example_explanation[
    "recommended_investigation"
]:
    print(
        " -",
        action
    )

print(
    "\nDecision-use note:"
)

print(
    example_explanation[
        "decision_use"
    ]
)

EARLYSIGNAL AI — EXPLAINED RISK ALERT

Record ID: ES-006044
Programme: Nutrition Support Programme
Location: Gwoza - Gwoza Site 3
Predicted Risk: High
High-Risk Model Score: 1.0000

Why this record was flagged:
The strongest operational signals associated with the High-Risk model score are Supply delay, Reporting delay and Low achievement.

Recommended investigation:
 - Review procurement, logistics and supply-chain bottlenecks.
 - Validate reporting timeliness and follow up on delayed field submissions.
 - Review progress against targets and identify implementation bottlenecks.

Decision-use note:
Recommended for human review; the model does not make operational decisions.


## Generate explanations for all held-out records

In [18]:
# ============================================================
# 18. Generate explanations for all held-out records
# ============================================================

explanation_records = []

for row_index in explanation_df.index:

    record = explanation_df.loc[
        row_index
    ]

    toward_high, away_from_high = (
        explain_high_risk_record(
            row_index,
            top_n=3
        )
    )

    toward_features = (
        toward_high[
            "feature"
        ].tolist()
    )

    toward_drivers = (
        toward_high[
            "driver"
        ].tolist()
    )

    away_drivers = (
        away_from_high[
            "driver"
        ].tolist()
    )

    investigation_actions = [
        investigation_guidance[
            feature
        ]
        for feature in toward_features
    ]

    explanation_records.append({
        "record_id":
            record["record_id"],

        "reporting_date":
            record["reporting_date"],

        "programme_name":
            record["programme_name"],

        "sector":
            record["sector"],

        "state":
            record["state"],

        "lga":
            record["lga"],

        "community":
            record["community"],

        "predicted_risk":
            record["predicted_risk"],

        "high_risk_model_score":
            float(
                record["probability_high"]
            ),

        "primary_driver":
            (
                toward_drivers[0]
                if len(toward_drivers) >= 1
                else "No strong risk-increasing signal"
            ),

        "secondary_driver":
            (
                toward_drivers[1]
                if len(toward_drivers) >= 2
                else ""
            ),

        "tertiary_driver":
            (
                toward_drivers[2]
                if len(toward_drivers) >= 3
                else ""
            ),

        "strongest_protective_signal":
            (
                away_drivers[0]
                if len(away_drivers) >= 1
                else ""
            ),

        "recommended_investigation":
            " | ".join(
                investigation_actions
            ),

        "decision_use":
            (
                "Recommended for human review; "
                "the model does not make "
                "operational decisions."
            )
    })


explainability_results = pd.DataFrame(
    explanation_records
)


print("=" * 72)
print("EXPLAINABILITY RESULTS GENERATED")
print("=" * 72)

print(
    "\nRows:",
    len(explainability_results)
)

print(
    "Columns:",
    len(explainability_results.columns)
)

print(
    "Duplicate record IDs:",
    int(
        explainability_results[
            "record_id"
        ].duplicated().sum()
    )
)

print(
    "Missing record IDs:",
    int(
        explainability_results[
            "record_id"
        ].isna().sum()
    )
)

print("\nPreview:")

display(
    explainability_results.head()
)

EXPLAINABILITY RESULTS GENERATED

Rows: 1680
Columns: 15
Duplicate record IDs: 0
Missing record IDs: 0

Preview:


,record_id,reporting_date,programme_name,sector,state,lga,community,predicted_risk,high_risk_model_score,primary_driver,secondary_driver,tertiary_driver,strongest_protective_signal,recommended_investigation,decision_use
0,ES-004826,2026-02-01,Nutrition Support Programme,Nutrition,Borno,Jere,Jere Site 1,Medium,1.176303e-03,Elevated complaints,Supply delay,Low achievement,Timely reporting,Review feedback and complaint records for recu...,Recommended for human review; the model does n...
1,ES-002138,2026-02-01,Livelihood Recovery Initiative,Livelihoods,Borno,Mafa,Mafa Site 1,High,9.815632e-01,Low achievement,Access constraints,Low staff availability,Lower complaints,Review progress against targets and identify i...,Recommended for human review; the model does n...
2,ES-005434,2026-02-01,Nutrition Support Programme,Nutrition,Borno,Bama,Bama Site 2,Low,1.100324e-21,Supply delay,Access constraints,Low staff availability,Strong achievement,"Review procurement, logistics and supply-chain...",Recommended for human review; the model does n...
3,ES-000666,2026-02-01,Community Resilience Programme,Protection/Resilience,Borno,Mafa,Mafa Site 3,Low,3.673446e-17,Low staff availability,Elevated complaints,Access constraints,Strong achievement,"Review staffing coverage, absences and field d...",Recommended for human review; the model does n...
4,ES-000058,2026-02-01,Community Resilience Programme,Protection/Resilience,Borno,Maiduguri,Maiduguri Site 2,Low,2.345190e-15,Low staff availability,Low activity completion,Supply delay,Strong achievement,"Review staffing coverage, absences and field d...",Recommended for human review; the model does n...


## Validate explanation coverage

In [19]:
# ============================================================
# 19. Validate explanation coverage
# ============================================================

high_risk_explanations = (
    explainability_results[
        explainability_results[
            "predicted_risk"
        ] == "High"
    ]
)

medium_risk_explanations = (
    explainability_results[
        explainability_results[
            "predicted_risk"
        ] == "Medium"
    ]
)

low_risk_explanations = (
    explainability_results[
        explainability_results[
            "predicted_risk"
        ] == "Low"
    ]
)


print("=" * 72)
print("EXPLANATION COVERAGE VALIDATION")
print("=" * 72)

print(
    "\nHigh-Risk records:",
    len(high_risk_explanations)
)

print(
    "Medium-Risk records:",
    len(medium_risk_explanations)
)

print(
    "Low-Risk records:",
    len(low_risk_explanations)
)

print(
    "\nHigh-Risk records with primary driver:",
    int(
        high_risk_explanations[
            "primary_driver"
        ].notna().sum()
    )
)

print(
    "High-Risk records with investigation guidance:",
    int(
        high_risk_explanations[
            "recommended_investigation"
        ].str.len().gt(0).sum()
    )
)

print(
    "\nUnique primary drivers:"
)

print(
    explainability_results[
        "primary_driver"
    ].value_counts()
)

EXPLANATION COVERAGE VALIDATION

High-Risk records: 140
Medium-Risk records: 615
Low-Risk records: 925

High-Risk records with primary driver: 140
High-Risk records with investigation guidance: 140

Unique primary drivers:
primary_driver
Low achievement                       402
Low activity completion               348
Supply delay                          277
Reporting delay                       212
Elevated complaints                   158
Access constraints                    152
Low staff availability                 77
Lower data quality                     30
Previous-month achievement pattern     21
Budget utilisation pattern              2
No strong risk-increasing signal        1
Name: count, dtype: int64


## Inspect highest-priority explained High-Risk cases

In [20]:
# ============================================================
# 20. Highest-priority explained High-Risk cases
# ============================================================

highest_explained_risk = (
    explainability_results[
        explainability_results[
            "predicted_risk"
        ] == "High"
    ]
    .sort_values(
        "high_risk_model_score",
        ascending=False
    )
    .head(15)
)


print("=" * 75)
print("HIGHEST-PRIORITY EXPLAINED HIGH-RISK CASES")
print("=" * 75)

display(
    highest_explained_risk[
        [
            "record_id",
            "reporting_date",
            "programme_name",
            "lga",
            "community",
            "high_risk_model_score",
            "primary_driver",
            "secondary_driver",
            "tertiary_driver"
        ]
    ]
)

HIGHEST-PRIORITY EXPLAINED HIGH-RISK CASES


,record_id,reporting_date,programme_name,lga,community,high_risk_model_score,primary_driver,secondary_driver,tertiary_driver
676,ES-006044,2026-04-01,Nutrition Support Programme,Gwoza,Gwoza Site 3,1.0,Supply delay,Reporting delay,Low achievement
1482,ES-002880,2026-08-01,Livelihood Recovery Initiative,Dikwa,Dikwa Site 6,1.0,Supply delay,Low achievement,Low activity completion
160,ES-002426,2026-02-01,Livelihood Recovery Initiative,Bama,Bama Site 4,1.0,Supply delay,Low achievement,Low activity completion
1213,ES-007231,2026-07-01,WASH Resilience Initiative,Ngala,Ngala Site 4,1.0,Supply delay,Access constraints,Reporting delay
680,ES-006716,2026-04-01,WASH Resilience Initiative,Konduga,Konduga Site 6,1.0,Low achievement,Reporting delay,Low activity completion
354,ES-002363,2026-03-01,Livelihood Recovery Initiative,Bama,Bama Site 2,1.0,Supply delay,Reporting delay,Low activity completion
485,ES-002460,2026-04-01,Livelihood Recovery Initiative,Bama,Bama Site 5,1.0,Elevated complaints,Reporting delay,Supply delay
369,ES-002331,2026-03-01,Livelihood Recovery Initiative,Bama,Bama Site 1,1.0,Supply delay,Reporting delay,Low achievement
1505,ES-004320,2026-08-01,Education Access Project,Dikwa,Dikwa Site 3,1.0,Low achievement,Elevated complaints,Supply delay
1195,ES-006494,2026-06-01,WASH Resilience Initiative,Jere,Jere Site 5,1.0,Low achievement,Low activity completion,Supply delay


## Check explanation diversity

In [21]:
# ============================================================
# 21. Explanation diversity audit
# ============================================================

high_primary_driver_counts = (
    high_risk_explanations[
        "primary_driver"
    ]
    .value_counts()
)

high_secondary_driver_counts = (
    high_risk_explanations[
        "secondary_driver"
    ]
    .value_counts()
)


print("=" * 72)
print("HIGH-RISK EXPLANATION DIVERSITY")
print("=" * 72)

print(
    "\nPrimary driver distribution:"
)

print(
    high_primary_driver_counts
)

print(
    "\nSecondary driver distribution:"
)

print(
    high_secondary_driver_counts
)

print(
    "\nUnique High-Risk primary drivers:",
    high_risk_explanations[
        "primary_driver"
    ].nunique()
)

HIGH-RISK EXPLANATION DIVERSITY

Primary driver distribution:
primary_driver
Low achievement            85
Supply delay               23
Low activity completion    15
Reporting delay            11
Elevated complaints         6
Name: count, dtype: int64

Secondary driver distribution:
secondary_driver
Low activity completion    52
Low achievement            29
Reporting delay            24
Supply delay               16
Access constraints         12
Elevated complaints         6
Lower data quality          1
Name: count, dtype: int64

Unique High-Risk primary drivers: 5


## Save explainability output

In [22]:
# ============================================================
# 22. Save standardized explainability output
# ============================================================

EXPLAINABILITY_RESULTS_PATH = (
    OUTPUT_DIR
    / "explainability_results.csv"
)

explainability_results.to_csv(
    EXPLAINABILITY_RESULTS_PATH,
    index=False
)


print("=" * 72)
print("EARLYSIGNAL AI — EXPLAINABILITY OUTPUT SAVED")
print("=" * 72)

print("\nSaved to:")
print(
    EXPLAINABILITY_RESULTS_PATH
)

print(
    "\nFile exists:",
    EXPLAINABILITY_RESULTS_PATH.exists()
)

print(
    "Rows saved:",
    len(explainability_results)
)

print(
    "Columns saved:",
    len(explainability_results.columns)
)

EARLYSIGNAL AI — EXPLAINABILITY OUTPUT SAVED

Saved to:
C:\Users\Ezekiel Mbaya\myenv\EarlySignal_AI\outputs\explainability_results.csv

File exists: True
Rows saved: 1680
Columns saved: 15


## Final Notebook 06 summary

In [23]:
# ============================================================
# 23. Final Explainable AI summary
# ============================================================

print("=" * 74)
print("EARLYSIGNAL AI — EXPLAINABLE AI SUMMARY")
print("=" * 74)

print(
    f"Records explained: "
    f"{len(explainability_results):,}"
)

print(
    f"Predicted High-Risk records explained: "
    f"{len(high_risk_explanations):,}"
)

print(
    f"Predicted Medium-Risk records explained: "
    f"{len(medium_risk_explanations):,}"
)

print(
    f"Predicted Low-Risk records explained: "
    f"{len(low_risk_explanations):,}"
)

print(
    "\nExplanation method: "
    "exact Logistic Regression feature contributions"
)

print(
    f"Transformed model features audited: "
    f"{len(transformed_feature_names)}"
)

print(
    f"Manager-facing operational features: "
    f"{len(operational_transformed_features)}"
)

print(
    "\nPrediction reproducibility: "
    f"{prediction_match.mean():.2%}"
)

print(
    "Contribution reconstruction audit:",
    "PASSED"
    if maximum_score_difference < 1e-10
    else "CHECK REQUIRED"
)

print(
    "Explainability leakage audit:",
    "PASSED"
    if len(leakage_overlap) == 0
    else "FAILED"
)

print(
    "\nInterpretation note: manager-facing explanations "
    "prioritize operational indicators rather than "
    "presenting location identifiers as causal drivers."
)

print(
    "Validation note: explanations describe a model "
    "validated on synthetic monitoring data."
)

print(
    "Decision-use note: explanations and investigation "
    "guidance support human review and do not automate "
    "programme decisions."
)

print("\nNotebook 06 complete.")

EARLYSIGNAL AI — EXPLAINABLE AI SUMMARY
Records explained: 1,680
Predicted High-Risk records explained: 140
Predicted Medium-Risk records explained: 615
Predicted Low-Risk records explained: 925

Explanation method: exact Logistic Regression feature contributions
Transformed model features audited: 77
Manager-facing operational features: 10

Prediction reproducibility: 100.00%
Contribution reconstruction audit: PASSED
Explainability leakage audit: PASSED

Interpretation note: manager-facing explanations prioritize operational indicators rather than presenting location identifiers as causal drivers.
Validation note: explanations describe a model validated on synthetic monitoring data.
Decision-use note: explanations and investigation guidance support human review and do not automate programme decisions.

Notebook 06 complete.
